# LLM-based synaptic type classification (excitatory / inhibitory)

1. Loads `Metadata_Neurons(manual).csv`, keeps only **3k neurons** (via CATMAID annotation `mw brain and inputs`).
2. For each neuron, calls Claude with a detailed prompt; optional web search (api_params); **strict JSON** output.
3. **Confidence** (HIGH/MEDIUM/LOW), **evidence**, and optional **ensemble** voting for uncertain cases.
4. Result DataFrame: `neuron_id`, `synapse_type`, `confidence`, `evidence`, `votes`, `ensemble_used`, **`search_urls`**, **`search_titles`** (lists when web search was used).
5. **Usage & cost**: token counts (input, output, cached/cache read) and web search requests are aggregated; after the run, estimated cost (USD) is printed per Anthropic pricing.

## Imports

In [1]:
import json
import re
import time
import pandas as pd
import anthropic
from tqdm.notebook import tqdm
from collections import Counter
import warnings
import os
warnings.filterwarnings("ignore")

os.chdir("../..")

# Sorry for this shitty path variables, It will be fixed on server soon (fix for this error: "Permission denied: '/root/.cargo/bin'")
os.environ["PATH"] = ":".join(
    p for p in os.environ["PATH"].split(":")
    if not p.startswith("/root")
)


import pymaid

catmaid_url = 'https://l1em.catmaid.virtualflybrain.org'
http_user = None
http_password = None
project_id = 1

rm = pymaid.CatmaidInstance(catmaid_url, http_user, http_password, project_id)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


## Load neuron metadata (3k via CATMAID)

In [2]:
skids_3k = sorted(map(int, pymaid.get_skids_by_annotation("mw brain and inputs")))

csv_path = "./Datasets/Generated/Metadata_Neurons(manual).csv"
df = pd.read_csv(csv_path)
df = df.fillna("")
cols = ["neuron_id", "name", "n_nodes", "n_connectors", "n_branches", "n_leafs", "cable_length", "annotation"]
df = df[df["neuron_id"].isin(skids_3k)][cols].copy()

df = df[1800:2000]

df

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


,neuron_id,name,n_nodes,n_connectors,n_branches,n_leafs,cable_length,annotation
2926,17360262,DALv1_PB2_left,1646,168,106,113,272913.88,"['mw interhemispheric integrator hop-3', 'mw i..."
2927,15590793,DL-LH_cand9 like left,881,67,60,62,164553.10,"['downstream of mPN iACT A2 left', 'mw gustato..."
2928,17360266,DALv1 contralateral 4 left,1326,116,60,65,266072.00,"['VLPm com*', '1_level-1_clusterID-1_signal-fl..."
2929,15607180,MN-R-Sens-B2-VM-07,1398,112,22,25,122362.47,"['Miroschnikow et al inputs and outputs', '0_l..."
2931,16131474,SY: weak on PVL09 & MBE24 (LNM_downstream3)_right,3015,130,86,93,323571.12,"['Larisa PVL09 downstream', 'mw gustatory-exte..."
...,...,...,...,...,...,...,...,...
3201,15575327,AN-R-Sens-B1-AVa-26,801,31,14,15,86482.86,"['monosynaptic to IPC-like', 'monosynaptic to ..."
3203,20556072,DPM descending thin left,1843,114,74,79,288534.80,"['YYCWMLEH', 'CCWA09Input', 'lv dVNC to T3+A1'..."
3205,8710444,BLAv1_2_right,2227,214,157,162,288938.12,"['BLAv1/2 a', '1_level-1_clusterID-1_signal-fl..."
3206,13429045,AF4: TOH right; Large Spider Neuron right,2670,162,94,96,250267.03,"['2_level-2_clusterID-6_signal-flow_-0.442', '..."


## API key

In [3]:
ANTHROPIC_API_KEY = ""

## System prompt (from FIXED — full)

In [4]:
SYSTEM_PROMPT = """
You are an expert in Drosophila Larval neuroanatomy and connectomics. Your task is to classify individual neurons from the first-instar Drosophila larval brain connectome as either **excitatory** or **inhibitory** based on all available metadata (name, morphology, and annotations).

## Background and data source

**Connectome and dataset:** The data come from the complete synaptic-resolution connectome of the Drosophila melanogaster first-instar larval brain (Winding, Pedigo et al., Science 2023, doi:10.1126/science.add9330). This connectome comprises 3016 neurons and ~548,000 synapses. Neurons and synapses were reconstructed in a nanometer-resolution electron microscopy (EM) volume of the central nervous system (CNS) using CATMAID (Collaborative Annotation Toolkit for Massive Amounts of Image Data)—an open-source tool for collaborative reconstruction and annotation of neuronal circuits from EM data (catmaid.readthedocs.io). The larval brain dataset is hosted and described as the L1 EM Catmaid Dataset on Virtual Fly Brain (VFB; see virtualflybrain.org blog/releases/catmaid/l1em/). It was produced by Albert Cardona's lab at HHMI Janelia and collaborators. The same dataset and naming/annotation conventions are used in the Science 2023 paper and in public CATMAID/L1EM resources.

**Neuron types and brain organization:** In the larval brain, neurons were clustered into 93 connectivity-based types; many cell classes are known (sensory neurons, projection neurons, Kenyon cells, MBONs, MBINs/DANs, local neurons, descending neurons, etc.). In Drosophila, neurotransmitter identity and thus excitatory vs inhibitory character are often linked to brain region and cell type: e.g. cholinergic neurons are typically excitatory; GABAergic and certain other populations are inhibitory. Brain regions and neuron naming conventions often reflect anatomy (neuropil, hemisphere, lineage), and neurons in the same region or with similar names often share functional and neurotransmitter profiles. You may use your knowledge of Drosophila neurobiology, published connectome papers (including the Science 2023 paper and related L1/larval literature), and, if helpful, general neurobiological principles or online sources to inform your classification—but you must base your final answer on the specific fields provided for each neuron.

## Reference material (from the official sources below—use this when classifying)

Before starting solving this problem explore all the references using search in web to understand all the context.

**1) Science paper — The connectome of an insect brain (doi:10.1126/science.add9330):**
Complete synaptic-resolution connectome of the Drosophila larva brain: 3016 neurons, ~548,000 synapses. Reconstruction was done in a nanometer-resolution EM volume of the CNS using CATMAID. Neurons were clustered into 93 connectivity-based types; morphology within clusters was similar and known functions matched clusters (e.g. olfactory PNs, KCs, MBINs/MBONs). Brain inputs: sensory neurons (SNs) and ascending neurons (ANs). Outputs: ring gland neurons (RGNs), DNs to SEZ (DNsSEZ), DNs to VNC (DNsVNC). Four connection types: axo-dendritic (a-d, majority), axo-axonic (a-a), dendro-dendritic (d-d), dendro-axonic (d-a). In adult Drosophila, a-a connections between otherwise excitatory (cholinergic) KCs can be inhibitory due to inhibitory mAChR-B in axon terminals; cholinergic neurons are typically excitatory. DANs (dopaminergic), MBONs, MBINs, KCs, local neurons (LNs), projection neurons (PNs)—naming and region help infer neurotransmitter and thus excitatory vs inhibitory.

**2) CATMAID documentation (catmaid.readthedocs.io):**
CATMAID = Collaborative Annotation Toolkit for Massive Amounts of Image Data. Features: fast terabyte-scale image browsing; collaborative microcircuit reconstruction and annotation; flexible hierarchical semantic annotation; multiple linked image stack display; Neuron Catalog; SVG and WebGL-based neuronal morphology viewer; open source (GPLv3). Annotations are free-text or hierarchical tags added by users during tracing; they document what tracers observe (e.g. neuron type, region, synapse character, notes). Use annotations as direct but uncurated evidence from expert tracers.

**3) L1 EM Catmaid Dataset on Virtual Fly Brain (virtualflybrain.org/blog/releases/catmaid/l1em/):**
Drosophila Larval (L1) CNS tracing by Albert Cardona's lab at HHMI Janelia Research Campus and collaborators. Project: "L1 CNS"; stack: "L1 CNS 0-tilt" — L1 CNS ssTEM at 3.8×3.8×50 nm of whole Drosophila 1st instar larval CNS. This is the same dataset as in the Science 2023 connectome paper; neuron names and annotations follow the same conventions used in the VFB L1EM CATMAID server and API.

## Input fields

You will receive for each neuron:
- **name**: The neuron's name or label (location, lineage, or type)
- **n_nodes**: Number of nodes (segments) in the neuron's morphology
- **n_connectors**: Number of synaptic connectors
- **n_branches**: Number of branches in the arbor
- **n_leafs**: Number of leaf (terminal) segments
- **cable_length**: Total cable length of the neuron
- **annotation**: Free-text annotations from CATMAID tracers (may include neurotransmitter hints, cell type info, etc.)

## Classification approach

Consider the **whole picture**: firstly - name (location/type) and annotation text, secondary - morphology (n_nodes, n_connectors, n_branches, n_leafs, cable_length). Aggregate all factors together—do not rely on a single cue. Give the most weight to **name** (because it usually encodes position and type, and brain regions in the fly larva often group neurons of similar synaptic character) and to **annotation** (because it reflects direct observer notes from EM, including possible synapse-type or neurotransmitter hints). Account for possible annotation noise and tracer errors: annotations are unverified; they may contain typos, technical jargon, or irrelevant tags—weigh explicit neurotransmitter/synapse-type mentions highly and discount inconsistent or noisy tags. Prefer consistent or biologically meaningful signals over single ambiguous tokens. If you would need to look up a brain region or cell type or lineage identifiers to be more accurate, you may do so using web search tool to map names/annotations to inhibitory vs excitatory; your answer must still be based only on the given fields and general Drosophila larval connectome knowledge. Be carefull with general knowledge data in your reasoning, use more speciffic knowledge (not this resources: "Generally in Drosophila larval neurons are ...", use instead this resources "The neuron ... uses ... neurotransmitter, since that ...")

1. **Assess confidence**: Based on all the data determine if you have HIGH, MEDIUM, or LOW confidence:
   - HIGH: Multiple clear indicators says the same information (e.g., "cholinergic", "GABAergic" in annotation, or well-known cell type in name)
   - MEDIUM: Some indicators but ambiguous
   - LOW: Unclear or conflicting signals

2. **For HIGH confidence**: Classify immediately.

3. **For MEDIUM/LOW confidence**: use additional web search:
   - Specific neuron class/type
   - Neurotransmitter identity in Drosophila larval brain literature
   - Drosophila larval region characteristics

## Output format (STRICT — JSON only)

You must respond with a single JSON object and nothing else. The object must have exactly these keys:
- "confidence": one of "HIGH", "MEDIUM", "LOW"
- "answer": one of "excitatory", "inhibitory"
- "evidence": a brief one-sentence summary (or empty string if none)

Use only Drosophila larval (not adult) information. No additional text, explanations, or keys.
"""

## User prompt template

In [5]:
USER_PROMPT_TEMPLATE = """
Classify this neuron. Respond with a single JSON object with 3 string kys: "confidence": "...", "answer": "...", "evidence": "..."

name: {name}
n_nodes: {n_nodes}
n_connectors: {n_connectors}
n_branches: {n_branches}
n_leafs: {n_leafs}
cable_length: {cable_length}
annotation: {annotation}
"""

## JSON schema for structured output + helpers

In [6]:
OUTPUT_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "confidence": {"type": "string", "enum": ["HIGH", "MEDIUM", "LOW"]},
        "answer": {"type": "string", "enum": ["excitatory", "inhibitory"]},
        "evidence": {"type": "string"},
    },
    "required": ["confidence", "answer", "evidence"],
    "additionalProperties": False,
}


def safe_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def parse_response_json(raw: str) -> dict:
    """Extract JSON object from response; return dict with confidence, answer, evidence."""
    out = {"confidence": "", "answer": "", "evidence": ""}
    raw = raw.strip()
    m = re.search(r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", raw, re.DOTALL)
    if not m:
        return out
    try:
        data = json.loads(m.group(0))
        out["confidence"] = str(data.get("confidence", "")).upper()
        if out["confidence"] not in ("HIGH", "MEDIUM", "LOW"):
            out["confidence"] = ""
        out["answer"] = str(data.get("answer", "")).lower()
        if out["answer"] not in ("excitatory", "inhibitory"):
            out["answer"] = ""
        out["evidence"] = str(data.get("evidence", "")).strip()
    except (json.JSONDecodeError, TypeError):
        pass
    return out


def is_valid_classification(result: dict) -> bool:
    return (
        result.get("confidence") in ("HIGH", "MEDIUM", "LOW")
        and result.get("answer") in ("excitatory", "inhibitory")
    )

## Classification: api_params (web search) + tool_use loop placeholder

In [7]:
def _extract_web_search_urls_titles(content_blocks) -> tuple[list, list]:
    """From msg.content extract all url and title from WebSearchToolResultBlock. Returns (urls, titles)."""
    urls, titles = [], []
    for block in content_blocks:
        if getattr(block, "type", None) != "web_search_tool_result":
            continue
        sub = getattr(block, "content", None)
        if not sub:
            continue
        for item in sub if isinstance(sub, list) else [sub]:
            if getattr(item, "type", None) == "web_search_result":
                u = getattr(item, "url", None)
                t = getattr(item, "title", None)
                if u:
                    urls.append(str(u))
                if t:
                    titles.append(str(t))
    return urls, titles


def _extract_usage(msg) -> dict:
    """Extract usage from Message for cost tracking."""
    u = getattr(msg, "usage", None) or {}
    out = {
        "input_tokens": getattr(u, "input_tokens", 0) or 0,
        "output_tokens": getattr(u, "output_tokens", 0) or 0,
        "cache_creation_input_tokens": getattr(u, "cache_creation_input_tokens", 0) or 0,
        "cache_read_input_tokens": getattr(u, "cache_read_input_tokens", 0) or 0,
        "web_search_requests": 0,
    }
    st = getattr(u, "server_tool_use", None)
    if st:
        out["web_search_requests"] = getattr(st, "web_search_requests", 0) or 0
    return out


def call_claude_single(
    name: str,
    n_nodes: str,
    n_connectors: str,
    n_branches: str,
    n_leafs: str,
    cable_length: str,
    annotation: str,
    api_key: str,
    allow_search: bool = True,
) -> dict:
    """
    Single Claude API call. Returns dict: confidence, answer, evidence, used_search,
    search_urls, search_titles, usage.
    """
    if not api_key:
        return {
            "confidence": "ERROR", "answer": "", "evidence": "No API key",
            "used_search": False, "search_urls": [], "search_titles": [], "usage": {"input_tokens": 0, "output_tokens": 0, "cache_creation_input_tokens": 0, "cache_read_input_tokens": 0, "web_search_requests": 0},
        }
    try:
        client = anthropic.Anthropic(api_key=api_key)
        system_messages = [
            {"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}}
        ]
        user_content = USER_PROMPT_TEMPLATE.format(
            name=name or "",
            n_nodes=n_nodes or "",
            n_connectors=n_connectors or "",
            n_branches=n_branches or "",
            n_leafs=n_leafs or "",
            cable_length=cable_length or "",
            annotation=annotation or "(none)",
        )

        tools = None
        if allow_search:
            tools = [{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}]

        # print(system_messages, user_content)

        api_params = {
            "model": "claude-sonnet-4-5-20250929",
            "max_tokens": 512,
            "system": system_messages,
            "messages": [{"role": "user", "content": user_content}],
            "output_config": {
                "format": {"type": "json_schema", "schema": OUTPUT_JSON_SCHEMA},
            },
        }
        if tools:
            api_params["tools"] = tools

        msg = client.messages.create(**api_params)
        search_urls, search_titles = _extract_web_search_urls_titles(msg.content)
        used_search = len(search_urls) > 0

        raw_text = ""
        for block in msg.content:
            if hasattr(block, "text") and block.text:
                raw_text += block.text

        # print(raw_text)

        result = parse_response_json(raw_text)
        result["used_search"] = used_search
        result["search_urls"] = search_urls
        result["search_titles"] = search_titles
        result["usage"] = _extract_usage(msg)
        return result
    except Exception as e:
        return {
            "confidence": "ERROR", "answer": "", "evidence": str(e)[:200],
            "used_search": False, "search_urls": [], "search_titles": [],
            "usage": {"input_tokens": 0, "output_tokens": 0, "cache_creation_input_tokens": 0,
                      "cache_read_input_tokens": 0, "web_search_requests": 0},
        }


def classify_with_ensemble(
    name: str,
    n_nodes: str,
    n_connectors: str,
    n_branches: str,
    n_leafs: str,
    cable_length: str,
    annotation: str,
    api_key: str,
    n_attempts: int = 3,
    allow_search: bool = True,
) -> dict:
    """Returns dict: synapse_type, confidence, votes, evidence, ensemble_used, search_urls, search_titles, usage."""
    def _merge_usage(usages: list) -> dict:
        return {
            "input_tokens": sum(u.get("input_tokens", 0) for u in usages),
            "output_tokens": sum(u.get("output_tokens", 0) for u in usages),
            "cache_creation_input_tokens": sum(u.get("cache_creation_input_tokens", 0) for u in usages),
            "cache_read_input_tokens": sum(u.get("cache_read_input_tokens", 0) for u in usages),
            "web_search_requests": sum(u.get("web_search_requests", 0) for u in usages),
        }

    result1 = call_claude_single(
        name, n_nodes, n_connectors, n_branches, n_leafs, cable_length, annotation,
        api_key, allow_search,
    )
    u1 = result1.get("usage") or {}
    urls1 = result1.get("search_urls") or []
    titles1 = result1.get("search_titles") or []
    if not is_valid_classification(result1):
        return {
            "synapse_type": "ERROR", "confidence": "ERROR", "votes": "0/1",
            "evidence": result1.get("evidence", "Invalid response"), "ensemble_used": False,
            "search_urls": urls1, "search_titles": titles1, "usage": u1,
        }
    if result1["confidence"] == "HIGH":
        return {
            "synapse_type": result1["answer"], "confidence": "HIGH", "votes": "1/1",
            "evidence": result1.get("evidence", ""), "ensemble_used": False,
            "search_urls": urls1, "search_titles": titles1, "usage": u1,
        }
    if n_attempts <= 1:
        return {
            "synapse_type": result1["answer"], "confidence": result1["confidence"], "votes": "1/1",
            "evidence": result1.get("evidence", ""), "ensemble_used": False,
            "search_urls": urls1, "search_titles": titles1, "usage": u1,
        }
    all_results = [result1]
    for _ in range(n_attempts - 1):
        time.sleep(0.3)
        r = call_claude_single(
            name, n_nodes, n_connectors, n_branches, n_leafs, cable_length, annotation,
            api_key, allow_search,
        )
        if is_valid_classification(r):
            all_results.append(r)
    answers = [x["answer"] for x in all_results]
    counts = Counter(answers)
    most_common = counts.most_common(1)[0][0]
    evidences = [x.get("evidence", "") for x in all_results if x.get("evidence")]
    all_urls = []
    all_titles = []
    for x in all_results:
        all_urls.extend(x.get("search_urls") or [])
        all_titles.extend(x.get("search_titles") or [])
    return {
        "synapse_type": most_common,
        "confidence": "REVIEW_NEEDED" if len(counts) > 1 else all_results[0]["confidence"],
        "votes": f"{counts[most_common]}/{len(all_results)}",
        "evidence": "; ".join(set(evidences)),
        "ensemble_used": True,
        "search_urls": all_urls,
        "search_titles": all_titles,
        "usage": _merge_usage([x.get("usage") or {} for x in all_results]),
    }

## Run classification

In [8]:
ALLOW_SEARCH = True
USE_ENSEMBLE = False
ENSEMBLE_ATTEMPTS = 1

In [9]:
# Accumulate usage across all neurons for cost report
total_usage = {"input_tokens": 0, "output_tokens": 0, "cache_creation_input_tokens": 0, "cache_read_input_tokens": 0, "web_search_requests": 0}

results = []
for i, row in tqdm(df.iterrows(), total=len(df), desc="Synapse type classification"):
    neuron_id = row["neuron_id"]
    name = safe_str(row["name"])
    n_nodes = safe_str(row["n_nodes"])
    n_connectors = safe_str(row["n_connectors"])
    n_branches = safe_str(row["n_branches"])
    n_leafs = safe_str(row["n_leafs"])
    cable_length = safe_str(row["cable_length"])
    annotation = safe_str(row["annotation"])

    if USE_ENSEMBLE:
        result = classify_with_ensemble(
            name=name,
            n_nodes=n_nodes,
            n_connectors=n_connectors,
            n_branches=n_branches,
            n_leafs=n_leafs,
            cable_length=cable_length,
            annotation=annotation,
            api_key=ANTHROPIC_API_KEY,
            n_attempts=ENSEMBLE_ATTEMPTS,
            allow_search=ALLOW_SEARCH,
        )
    else:
        r = call_claude_single(
            name=name, n_nodes=n_nodes, n_connectors=n_connectors,
            n_branches=n_branches, n_leafs=n_leafs, cable_length=cable_length,
            annotation=annotation, api_key=ANTHROPIC_API_KEY, allow_search=ALLOW_SEARCH,
        )
        result = {
            "synapse_type": r.get("answer", ""),
            "confidence": r.get("confidence", ""),
            "votes": "1/1",
            "evidence": r.get("evidence", ""),
            "ensemble_used": False,
            "search_urls": r.get("search_urls") or [],
            "search_titles": r.get("search_titles") or [],
            "usage": r.get("usage") or {},
        }

    u = result.get("usage") or {}
    for k in total_usage:
        total_usage[k] += u.get(k, 0)

    results.append({
        "neuron_id": neuron_id,
        "synapse_type": result["synapse_type"],
        "confidence": result["confidence"],
        "evidence": result["evidence"],
        "votes": result["votes"],
        "ensemble_used": result["ensemble_used"],
        "search_urls": result.get("search_urls") or [],
        "search_titles": result.get("search_titles") or [],
    })

result_df = pd.DataFrame(results)
result_df.head(20)

Synapse type classification:   0%|          | 0/200 [00:00<?, ?it/s]

,neuron_id,synapse_type,confidence,evidence,votes,ensemble_used,search_urls,search_titles
0,17360262,excitatory,MEDIUM,DALv1 neurons are local neurons in the central...,1/1,False,[https://www.frontiersin.org/journals/neurosci...,[Frontiers | Protocerebral Bridge Neurons That...
1,15590793,excitatory,MEDIUM,Neuron named DL-LH_cand9 is a third-order olfa...,1/1,False,[https://pmc.ncbi.nlm.nih.gov/articles/PMC6529...,[Neurogenetic dissection of the Drosophila lat...
2,17360266,excitatory,LOW,DALv1 is a brain lineage in the dorso-anterior...,1/1,False,[https://pmc.ncbi.nlm.nih.gov/articles/PMC7910...,[Anatomy and Neural Pathways Modulating Distin...
3,15607180,excitatory,HIGH,Pharyngeal and somatosensory neurons in Drosop...,1/1,False,[https://pmc.ncbi.nlm.nih.gov/articles/PMC5907...,[Neural circuits driving larval locomotion in ...
4,16131474,excitatory,MEDIUM,MB2INs (pre-MBINs) and LHNs in Drosophila larv...,1/1,False,[https://www.sdbonline.org/sites/fly/aignfam/b...,"[Drosophila gene families: Biogenic Amines, A ..."
5,17360282,excitatory,MEDIUM,DALv1_PB1_left is a central complex neuron wit...,1/1,False,[https://link.springer.com/article/10.1186/s13...,[Neural circuits driving larval locomotion in ...
6,7480731,excitatory,HIGH,Rh5 photoreceptors in Drosophila larvae releas...,1/1,False,[https://www.nature.com/articles/s41467-018-03...,[Dedicated photoreceptor pathways in Drosophil...
7,17360286,excitatory,MEDIUM,DALv1 is a contralateral brain lineage neuron ...,1/1,False,[https://pmc.ncbi.nlm.nih.gov/articles/PMC7910...,[Anatomy and Neural Pathways Modulating Distin...
8,9594273,excitatory,LOW,Neuron name indicates contralateral neuron in ...,1/1,False,[https://pmc.ncbi.nlm.nih.gov/articles/PMC4917...,[Comparative Neuroanatomy of the Lateral Acces...
9,17360296,inhibitory,MEDIUM,DALv1_PB4_left is a local neuron (mw LN annota...,1/1,False,[https://pmc.ncbi.nlm.nih.gov/articles/PMC3767...,[Development and plasticity of the Drosophila ...


## Summary and export

In [10]:
print(result_df["synapse_type"].value_counts(dropna=False))
print("\n")
print(result_df["confidence"].value_counts(dropna=False))
# print("Ensemble used:", result_df["ensemble_used"].sum())

result_df.to_csv("./Datasets/Generated/Syn_Types(By_Annotations_LLM)_10.csv", index=False)

synapse_type
excitatory    160
inhibitory     38
                2
Name: count, dtype: int64


confidence
MEDIUM    93
HIGH      85
LOW       20
           2
Name: count, dtype: int64


In [11]:
# --- Usage & cost (Anthropic Sonnet 4.5, official pricing) ---
# https://www.anthropic.com/pricing — Input $3/MTok, Output $15/MTok (≤200K)
# Cache: write $3.75/MTok, read $0.30/MTok. Web search: $10/1K searches = $0.01/search.

PRICE_INPUT_PER_MTOK = 3.0
PRICE_OUTPUT_PER_MTOK = 15.0
PRICE_CACHE_WRITE_PER_MTOK = 3.75
PRICE_CACHE_READ_PER_MTOK = 0.30
PRICE_WEB_SEARCH_PER_1K = 10.0  # $ per 1000 searches

in_tok = total_usage["input_tokens"]
out_tok = total_usage["output_tokens"]
cache_w = total_usage["cache_creation_input_tokens"]
cache_r = total_usage["cache_read_input_tokens"]
n_searches = total_usage["web_search_requests"]
# Standard input = total input minus cache read/write (they are billed separately)
std_input = in_tok - cache_r - cache_w
if std_input < 0:
    std_input = 0

cost_std_input = (std_input / 1e6) * PRICE_INPUT_PER_MTOK
cost_cache_write = (cache_w / 1e6) * PRICE_CACHE_WRITE_PER_MTOK
cost_cache_read = (cache_r / 1e6) * PRICE_CACHE_READ_PER_MTOK
cost_output = (out_tok / 1e6) * PRICE_OUTPUT_PER_MTOK
cost_web_search = (n_searches / 1000.0) * PRICE_WEB_SEARCH_PER_1K
total_cost_usd = cost_std_input + cost_cache_write + cost_cache_read + cost_output + cost_web_search

print("=== Usage (total over all neurons) ===")
print(f"  input_tokens (total):           {in_tok:,}")
print(f"  output_tokens:                  {out_tok:,}")
print(f"  cache_creation_input_tokens:    {cache_w:,}")
print(f"  cache_read_input_tokens:        {cache_r:,}")
print(f"  web_search_requests:           {n_searches:,}")
print()
print("=== Estimated cost (USD, Sonnet 4.5) ===")
print(f"  Standard input:   ${cost_std_input:.4f}")
print(f"  Cache write:      ${cost_cache_write:.4f}")
print(f"  Cache read:       ${cost_cache_read:.4f}")
print(f"  Output:           ${cost_output:.4f}")
print(f"  Web search:      ${cost_web_search:.4f}")
print(f"  TOTAL:            ${total_cost_usd:.4f}")

=== Usage (total over all neurons) ===
  input_tokens (total):           320,358
  output_tokens:                  58,413
  cache_creation_input_tokens:    9,905,355
  cache_read_input_tokens:        7,293,136
  web_search_requests:           629

=== Estimated cost (USD, Sonnet 4.5) ===
  Standard input:   $0.0000
  Cache write:      $37.1451
  Cache read:       $2.1879
  Output:           $0.8762
  Web search:      $6.2900
  TOTAL:            $46.4992


In [12]:
result_df["evidence"][2]

'DALv1 is a brain lineage in the dorso-anterior lateral protocerebrum with contralateral projections through the great commissure. Although the Winding Pedigo 2023 paper did not explicitly specify neurotransmitter type for individual DALv1 neurons, lineage studies indicate brain lineages can be cholinergic (excitatory), GABAergic (inhibitory), or glutamatergic (inhibitory). The DALv1 contralateral architecture and name suggest integrative function; cholinergic neurons are typically excitatory in larval brain.'